### Feature Matching Zip Codes and Town Names to master dataset

In [1]:
import pandas as pd
from difflib import get_close_matches

town_zip = pd.read_csv('datasets_address/Address_Datasets/towns_zips_in_ct.csv')
town_zip["Zip Code"] = town_zip["Zip Code"].astype(str).str.zfill(5)
print(town_zip.head())



  Zip Code        City
0    06230    Abington
1    06516  Allingtown
2    06231      Amston
3    06232     Andover
4    06401     Ansonia


In [ ]:
data_frame = pd.DataFrame({
    "City": ["Danbury", "Brookfield", "Dabury", "Stamford", "kladjfhgaidhfgakdhfga", "Bethel", "Standfort"],
    "Zip_code": ["06811", "06804", "06810", "06805", "06812", "12345", "06805"],
})

In [ ]:
# Verify Town Names
def verify_town(city, accepted_towns):
    validity = []
    for i in city:
        if i in accepted_towns.values:
            validity.append("True")
            print(f"{i} is a valid town name.")
        elif len(get_close_matches(i, accepted_towns, n=1, cutoff=0.8)) > 0:
            validity.append("Possibly")
            print(f"{i} is similar to a valid town name. {get_close_matches(i, accepted_towns, n=1, cutoff=0.8)[0]} is a valid town name.")
        else:
            validity.append("Needs Review")
            print(f"{i} is NOT a valid town name. Please check the spelling or formatting.")

    return validity


town_validity = verify_town(data_frame["City"], town_zip["City"])
data_frame["town_name_validity"] = town_validity

print(data_frame)


Danbury is a valid town name.
Brookfield is a valid town name.
Dabury is similar to a valid town name. Danbury is a valid town name.
Stamford is a valid town name.
kladjfhgaidhfgakdhfga is NOT a valid town name. Please check the spelling or formatting.
Bethel is a valid town name.
Standfort is NOT a valid town name. Please check the spelling or formatting.
                    City Zip_code town_name_validity
0                Danbury    06811               True
1             Brookfield    06804               True
2                 Dabury    06810           Possibly
3               Stamford    06805               True
4  kladjfhgaidhfgakdhfga    06812       Needs Review
5                 Bethel    12345               True
6              Standfort    06805       Needs Review


In [8]:
# Verify Zip Codes
def verify_zipcodes(zip_codes, accepted_zips):
    validity = []
    for i in zip_codes:
        if i in accepted_zips.values:
            validity.append("True")
            print(f"{i} is a valid zip code.")
        elif len(get_close_matches(i, accepted_zips, n=1, cutoff=0.8)) > 0:
            validity.append("Possibly")
            print(f"{i} is similar to a valid zip code. {get_close_matches(i, accepted_zips, n=1, cutoff=0.8)[0]} is a valid zip code.")
        else:
            validity.append("Needs Review")
            print(f"{i} is NOT a valid zip code. Please check the spelling or formatting.")

    return validity    


zip_validity = verify_zipcodes(data_frame["Zip_code"], town_zip["Zip Code"])
data_frame["zip_code_validity"] = zip_validity

print(data_frame)


06811 is a valid zip code.
06804 is a valid zip code.
06810 is a valid zip code.
06805 is similar to a valid zip code. 06905 is a valid zip code.
06812 is a valid zip code.
12345 is NOT a valid zip code. Please check the spelling or formatting.
                    City Zip_code town_name_validity zip_code_validity
0                Danbury    06811               True              True
1             Brookfield    06804               True              True
2                 Dabury    06810           Possibly              True
3               Stamford    06805               True          Possibly
4  kladjfhgaidhfgakdhfga    06812       Needs Review              True
5                 Bethel    12345               True      Needs Review


In [ ]:
# Making sure that both zip codes and towns match up with each other. If they don't, then the entry needs review
def verify_town_zip_match(city, zip_code, accepted_towns, accepted_zips):
    validity = []

    for i in zip_code.values:
        corresponding_town = city[zip_code == i].values[0]
        if i in accepted_zips.values:
            index = accepted_zips[accepted_zips == i].index[0]
            if corresponding_town == accepted_towns[index]:
                validity.append("Town Name and Zip Code Match")
                print(f"{i} and {corresponding_town} are a valid town and zip code match.")
            elif len(get_close_matches(corresponding_town, accepted_towns, n=1, cutoff=0.8)) > 0:
                validity.append("Town Name Needs Review but similar name matches zip code")
                print(f"{i} and {corresponding_town} are NOT a valid town and zip code match. {i} is a valid zip code, {get_close_matches(corresponding_town, accepted_towns, n=1, cutoff=0.8)[0]} is a valid town.")
                if get_close_matches(corresponding_town, accepted_towns, n=1, cutoff=0.8)[0] == accepted_towns[index]:
                    print(f"{i} and {get_close_matches(corresponding_town, accepted_towns, n=1, cutoff=0.8)[0]} are a valid town and zip code match.")
            else:
                validity.append("Town Name Needs Review and Does NOT Match Zip Code")
                print(f"{corresponding_town} is NOT a valid town name. Please check the spelling or formatting.")
        elif len(get_close_matches(i, accepted_zips, n=1, cutoff=0.8)) > 0:
            print(f"{i} is similar to a valid zip code. {get_close_matches(i, accepted_zips, n=1, cutoff=0.8)[0]} is a valid zip code.")
            if get_close_matches(i, accepted_zips, n=1, cutoff = 0.8)[0] in accepted_zips.values:
                index = accepted_zips[accepted_zips == get_close_matches(i, accepted_zips, n=1, cutoff=0.8)[0]].index[0]
                if corresponding_town == accepted_towns[index]:
                    validity.append("Zip Code needs review but Town Name is Valid and Matches")
                    print(f"{get_close_matches(i, accepted_zips, n=1, cutoff=0.8)[0]} and {corresponding_town} are a valid town and zip code match.")
                else:
                    validity.append("Zip Code needs review and Does NOT Match town name")
                    print(f"{corresponding_town} is NOT a valid town name. Please check the spelling or formatting.")
        else:
            if corresponding_town in accepted_towns.values:
                validity.append("Zip Code needs review")
                print(f"{corresponding_town} is a valid town name. {i} is NOT a valid zip code. Please check the spelling or formatting.")
            elif len(get_close_matches(corresponding_town, accepted_towns, n=1, cutoff=0.8)) > 0:
                validity.append("Zip Code needs review but Town Name is similar to a valid town name")
                print(f"{i} is NOT a valid zip code. {get_close_matches(corresponding_town, accepted_towns, n=1, cutoff=0.8)[0]} is a valid town name.")
            else:
                validity.append("Both Town Name and Zip Code need review")
                print(f"{i} is NOT a valid zip code. Please check the spelling or formatting.")

            
        print()
        
    return validity

validity_results = verify_town_zip_match(data_frame["City"], data_frame["Zip_code"], town_zip["City"], town_zip["Zip Code"])
data_frame["town_zip_match_validity"] = validity_results
display(data_frame)

06811 and Danbury are a valid town and zip code match.

06804 and Brookfield are a valid town and zip code match.

06810 and Dabury are NOT a valid town and zip code match. 06810 is a valid zip code, Danbury is a valid town.
06810 and Danbury are a valid town and zip code match.

06805 is similar to a valid zip code. 06905 is a valid zip code.
06905 and Stamford are a valid town and zip code match.

kladjfhgaidhfgakdhfga is NOT a valid town name. Please check the spelling or formatting.

Bethel is a valid town name. 12345 is NOT a valid zip code. Please check the spelling or formatting.

06805 is similar to a valid zip code. 06905 is a valid zip code.
06905 and Stamford are a valid town and zip code match.



,City,Zip_code,town_name_validity,town_zip_match_validity
0,Danbury,06811,True,Town Name and Zip Code Match
1,Brookfield,06804,True,Town Name and Zip Code Match
2,Dabury,06810,Possibly,Town Name Needs Review but similar name matche...
3,Stamford,06805,True,Zip Code needs review but Town Name is Valid a...
4,kladjfhgaidhfgakdhfga,06812,Needs Review,Town Name Needs Review and Does NOT Match Zip ...
5,Bethel,12345,True,Zip Code needs review
6,Standfort,06805,Needs Review,Zip Code needs review but Town Name is Valid a...
